# 03 — Data Exploration

Regenerates the figures from the thesis **Eksplorasi Data** section
(`methodology.tex` §Eksplorasi Data), inlined here. Same logic as
`documents/thesis_v6.1/latex_source/figures/*.py`, but **streams one date at a
time** (no ~40 GB full-stack load) and shows each figure inline.

Figures: CDL label map · CDL class-area distribution · valid-pixel coverage ·
RGB (peak + 25-date grid) · 10-band grid · NDVI (temporal / 25-date grid /
per-class) · spectral profiles · band correlation · per-class band histograms ·
S2+CDL patch detail.

> Reads raw S2 + CDL from T7. Set `SAVE=True` to also write PNGs to `OUT_DIR`.

In [ ]:
import os, re, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
from collections import Counter
warnings.filterwarnings('ignore')

# ── paths (thesis config.py) ──
S2_DIR   = '/Volumes/T7/research-crop-mapping-geoai/data/raw_v6/s2/2024'
CDL_PATH = '/Volumes/T7/research-crop-mapping-geoai/data/raw_v6/cdl/cdl_2024_study_area_filtered.tif'
OUT_DIR  = '/Users/dikaizm/Documents/PROGRAMMING/ml-ai/research-crop-mapping-thesis/research-crop-mapping-geoai/documents/thesis_v6.1/figures'
SAVE     = False   # True → also savefig to OUT_DIR

S2_BANDS = ['B2','B3','B4','B5','B6','B7','B8','B8A','B11','B12']
BAND_LABELS = {'B2':'Blue (490 nm)','B3':'Green (560 nm)','B4':'Red (665 nm)',
    'B5':'Red Edge 1 (705 nm)','B6':'Red Edge 2 (740 nm)','B7':'Red Edge 3 (783 nm)',
    'B8':'NIR (842 nm)','B8A':'Red Edge 4 (865 nm)','B11':'SWIR 1 (1610 nm)','B12':'SWIR 2 (2190 nm)'}
TRAIN_CLASSES = {1:'Corn',3:'Rice',24:'Winter Wheat',36:'Alfalfa',
    54:'Tomatoes',69:'Grapes',75:'Almonds',76:'Walnuts'}
CDL_COLORS = {0:'#d3d3d3',1:'#ffd400',3:'#00a9e6',24:'#a87000',36:'#ffa8e3',
    54:'#f5a27a',69:'#704489',75:'#00a884',76:'#ebd6b0'}
PEAK_DATE = '2024_07_14'
plt.rcParams.update({'font.size':10,'figure.dpi':110})

# ── file index + streaming loaders ──
FILES = sorted(f for f in os.listdir(S2_DIR) if f.endswith('.tif') and not f.startswith('._'))
DATES = [f.replace('S2H_2024_','').replace('.tif','') for f in FILES]   # 'YYYY_MM_DD'
DATE2FILE = {d: os.path.join(S2_DIR, f) for d, f in zip(DATES, FILES)}

def read_date(date, bands=None, ds=1):
    """Read one date → (H,W,Bsel) float32 + nodata mask. bands=list of names or None(all)."""
    path = DATE2FILE[date]
    with rasterio.open(path) as src:
        if ds > 1:
            arr = src.read(out_shape=(src.count, src.height//ds, src.width//ds)).astype(np.float32)
        else:
            arr = src.read().astype(np.float32)
        tf, crs = src.transform, src.crs
    arr = arr.transpose(1, 2, 0)                       # (H,W,B)
    nodata = (arr[:,:,0] <= -9990) | (arr[:,:,0] == 0) | np.isnan(arr).any(-1)
    if bands is not None:
        arr = arr[:, :, [S2_BANDS.index(b) for b in bands]]
    return arr, nodata, tf, crs

_CDL_CACHE = {}
def load_cdl_grid(shape, transform, crs):
    """Reproject CDL_PATH onto the (shape,transform,crs) S2 grid (cached)."""
    key = (shape, str(transform))
    if key in _CDL_CACHE: return _CDL_CACHE[key]
    with rasterio.open(CDL_PATH) as src:
        cdl = src.read(1).astype(np.int32); s_tf, s_crs = src.transform, src.crs
    dest = np.zeros(shape, np.int32)
    reproject(cdl, dest, src_transform=s_tf, src_crs=s_crs,
              dst_transform=transform, dst_crs=crs, resampling=Resampling.nearest)
    _CDL_CACHE[key] = dest; return dest

def show(fig, name):
    if SAVE:
        os.makedirs(OUT_DIR, exist_ok=True)
        fig.savefig(os.path.join(OUT_DIR, name), dpi=150, bbox_inches='tight'); print('saved', name)
    plt.show()

print(len(DATES), 'S2 dates:', DATES)

## 1. CDL label map (`cdl_label_map.png`)

In [ ]:
USDA = {0:'#d3d3d3',1:'#ffd400',3:'#00a9e6',24:'#a87000',36:'#ffa8e3',54:'#f5a27a',69:'#704489',75:'#00a884',76:'#ebd6b0'}
_, _, tf, crs = read_date(PEAK_DATE, bands=['B2'])
cdl = load_cdl_grid((read_date(PEAK_DATE, bands=['B2'])[0].shape[0], read_date(PEAK_DATE, bands=['B2'])[0].shape[1]), tf, crs)
ids = list(TRAIN_CLASSES); disp = np.zeros(cdl.shape, np.int32)
colors=[USDA[0]]; labels=['Background']
for k,cid in enumerate(ids,1):
    disp[cdl==cid]=k; colors.append(USDA[cid]); labels.append(TRAIN_CLASSES[cid])
n=len(colors); cmap=ListedColormap(colors); norm=BoundaryNorm(np.arange(-.5,n+.5),n)
H=disp.shape[0]; step=max(1,H//1500)
fig,ax=plt.subplots(figsize=(10,8))
ax.imshow(disp[::step,::step],cmap=cmap,norm=norm,interpolation='nearest')
ax.legend(handles=[mpatches.Patch(color=c,label=l) for c,l in zip(colors,labels)],
          loc='lower left',fontsize=7,ncol=3,framealpha=.9)
ax.set_title('Peta Label CDL 2024 — Sacramento Valley, California'); ax.axis('off')
show(fig,'cdl_label_map.png')

## 2. CDL class-area distribution (`cdl_class_distribution_area.png`)

In [ ]:
with rasterio.open(CDL_PATH) as src: raw = src.read(1).astype(np.int32)
cnt = Counter(raw.flatten()); ha_per_px = 77.1/10000.0
ks=list(TRAIN_CLASSES); names=[TRAIN_CLASSES[k] for k in ks]
ha=[cnt.get(k,0)*ha_per_px for k in ks]; cols=[CDL_COLORS[k] for k in ks]
o=sorted(range(len(ha)),key=lambda i:ha[i])
names=[names[i] for i in o]; ha=[ha[i] for i in o]; cols=[cols[i] for i in o]
fig,ax=plt.subplots(figsize=(10,5)); bars=ax.barh(names,ha,color=cols)
ax.set_xlabel('Area (hektar)'); ax.set_title('Distribusi Cakupan Area 8 Kelas Tanaman Target — CDL 2024')
for b,h in zip(bars,ha): ax.text(b.get_width()+200,b.get_y()+b.get_height()/2,f'{h:,.0f} ha',va='center',fontsize=9)
ax.margins(x=.15); fig.tight_layout(); show(fig,'cdl_class_distribution_area.png')

## 3. Valid-pixel coverage per date (`s2_data_coverage.png`)

In [ ]:
valid_pct=[]
for d in DATES:
    _,nd,_,_=read_date(d,bands=['B2'])
    valid_pct.append((1.0-nd.mean())*100)
valid_pct=np.array(valid_pct); T=len(DATES)
fig,ax=plt.subplots(figsize=(14,5))
cols=['#d62728' if v<50 else '#ff7f0e' if v<90 else '#1f77b4' for v in valid_pct]
ax.bar(range(T),valid_pct,color=cols)
ax.axhline(50,color='red',ls='--',lw=1.5,alpha=.7,label='Ambang 50%')
ax.set_xticks(range(T)); ax.set_xticklabels([d.replace('2024_','') for d in DATES],rotation=45,ha='right',fontsize=7)
ax.set_ylabel('Piksel Valid (%)'); ax.set_ylim(0,105); ax.legend(fontsize=9)
ax.set_title('Kualitas Data — Persentase Piksel Valid per Tanggal Akuisisi 2024')
for i,v in enumerate(valid_pct): ax.text(i,v+1,f'{v:.1f}%',ha='center',fontsize=6,rotation=90)
show(fig,'s2_data_coverage.png')

## 4. RGB — peak date + 25-date true-color grid (`s2_rgb_*.png`)

In [ ]:
def stretch(a):
    v=a[np.isfinite(a)];
    if v.size==0: return np.zeros_like(a)
    lo,hi=np.percentile(v,[2,98]); return np.clip((a-lo)/max(hi-lo,1e-6),0,1)
# peak-date single RGB
arr,nd,_,_=read_date(PEAK_DATE,bands=['B4','B3','B2'])
arr[nd]=np.nan; rgb=np.dstack([stretch(arr[:,:,i]) for i in range(3)])
fig,ax=plt.subplots(figsize=(8,8)); ax.imshow(rgb); ax.axis('off')
ax.set_title(f'True Color (B4/B3/B2) — {PEAK_DATE}'); show(fig,'s2_rgb_single.png')

In [ ]:
# 25-date RGB grid (5×5), downsampled
ncol=5; nrow=int(np.ceil(len(DATES)/ncol))
fig,axes=plt.subplots(nrow,ncol,figsize=(22,4.6*nrow))
for i,ax in enumerate(np.array(axes).flat):
    if i>=len(DATES): ax.axis('off'); continue
    a,nd,_,_=read_date(DATES[i],bands=['B4','B3','B2'],ds=8); a[nd]=np.nan
    ax.imshow(np.dstack([stretch(a[:,:,k]) for k in range(3)]))
    ax.set_title(DATES[i].replace('_','-'),fontsize=9); ax.axis('off')
plt.tight_layout(pad=.5); show(fig,'s2_rgb_grid.png')

## 5. 10-band grid, peak date (`s2_band_grid.png`)

In [ ]:
arr,nd,_,_=read_date(PEAK_DATE)
fig,axes=plt.subplots(2,5,figsize=(20,8))
for i,ax in enumerate(axes.flat):
    if i<len(S2_BANDS):
        bd=arr[:,:,i].copy(); bd[nd]=np.nan; vmin,vmax=np.nanpercentile(bd,[2,98])
        im=ax.imshow(bd,cmap='gray',vmin=vmin,vmax=vmax)
        ax.set_title(f'{S2_BANDS[i]}\n{BAND_LABELS[S2_BANDS[i]]}',fontsize=9)
        plt.colorbar(im,ax=ax,fraction=.046,pad=.04)
    ax.axis('off')
fig.suptitle(f'Komposit 10 Band Sentinel-2 — {PEAK_DATE}',fontsize=14); show(fig,'s2_band_grid.png')

## 6. NDVI — area-mean temporal profile (`s2_ndvi_temporal.png`)

In [ ]:
ts=[]
for d in DATES:
    a,nd,_,_=read_date(d,bands=['B8','B4'])
    nir,red=a[:,:,0],a[:,:,1]; m=(~nd)&((nir+red)>0)
    ts.append(np.nanmean((nir[m]-red[m])/(nir[m]+red[m])))
fig,ax=plt.subplots(figsize=(14,5)); x=np.arange(len(DATES))
ax.plot(x,ts,'o-',color='#2ca02c',lw=2,ms=8); ax.fill_between(x,ts,alpha=.15,color='#2ca02c')
ax.set_xticks(x); ax.set_xticklabels([d.replace('2024_','') for d in DATES],rotation=45,ha='right',fontsize=7)
ax.set_ylabel('NDVI (rata-rata area studi)'); ax.grid(alpha=.3)
ax.set_title('Profil Rata-rata NDVI Temporal — Sacramento Valley 2024'); show(fig,'s2_ndvi_temporal.png')

## 7. NDVI — 25-date grid (`s2_ndvi_grid.png`)

In [ ]:
ncol=5; nrow=int(np.ceil(len(DATES)/ncol))
fig,axes=plt.subplots(nrow,ncol,figsize=(20,3.4*nrow))
im=None
for i,ax in enumerate(np.array(axes).flat):
    if i>=len(DATES):
        ax.axis('off'); continue
    a,nd,_,_=read_date(DATES[i],bands=['B8','B4'],ds=8)
    ndvi=(a[:,:,0]-a[:,:,1])/(a[:,:,0]+a[:,:,1]+1e-6); ndvi[nd]=np.nan
    im=ax.imshow(ndvi,cmap='RdYlGn',vmin=-1,vmax=1,interpolation='bilinear')
    ax.set_title(DATES[i].replace('_','-'),fontsize=10); ax.set_xticks([]); ax.set_yticks([])
fig.subplots_adjust(bottom=.06,wspace=.02,hspace=.2)
cb=fig.colorbar(im,ax=list(np.array(axes).flat),orientation='horizontal',fraction=.03,pad=.04)
cb.set_label('NDVI'); show(fig,'s2_ndvi_grid.png')

## 8. NDVI — per-class temporal profile (`s2_ndvi_per_class.png`)

In [ ]:
a0,_,tf,crs=read_date(PEAK_DATE,bands=['B2'])
cdl=load_cdl_grid(a0.shape[:2],tf,crs)
masks={cid:(cdl==cid) for cid in TRAIN_CLASSES if (cdl==cid).sum()>=1000}
series={cid:[] for cid in masks}
for d in DATES:
    a,nd,_,_=read_date(d,bands=['B8','B4'])
    ndvi=np.full(nd.shape,np.nan,np.float32); v=(~nd)&((a[:,:,0]+a[:,:,1])>0)
    ndvi[v]=(a[v,0]-a[v,1])/(a[v,0]+a[v,1])
    for cid,mk in masks.items(): series[cid].append(np.nanmean(ndvi[mk]))
fig,ax=plt.subplots(figsize=(14,6)); x=np.arange(len(DATES)); cl=plt.cm.tab10(np.linspace(0,1,8))
for k,(cid,ys) in enumerate(series.items()):
    ax.plot(x,ys,'o-',color=cl[list(TRAIN_CLASSES).index(cid)],lw=1.8,ms=6,label=TRAIN_CLASSES[cid])
ax.set_xticks(x); ax.set_xticklabels([d.replace('2024_','') for d in DATES],rotation=45,ha='right',fontsize=7)
ax.set_ylabel('NDVI rata-rata'); ax.legend(loc='upper left',fontsize=8,ncol=4); ax.grid(alpha=.3)
ax.set_title('Profil NDVI Per Kelas Tanaman — Sacramento Valley 2024'); show(fig,'s2_ndvi_per_class.png')

## 9. Spectral profiles — best-NDVI date per quarter (`s2_spectral_profile.png`)

In [ ]:
QUARTAL={'Q1 (Jan–Mar)':(1,3),'Q2 (Apr–Jun)':(4,6),'Q3 (Jul–Sep)':(7,9),'Q4 (Okt–Des)':(10,12)}
month={d:int(d.split('_')[1]) for d in DATES}
def ndvi_mean(d):
    a,nd,_,_=read_date(d,bands=['B8','B4']); m=~nd
    return np.nanmean((a[m,0]-a[m,1])/(a[m,0]+a[m,1]+1e-8))
sel={}
for lab,(s,e) in QUARTAL.items():
    cand=[d for d in DATES if s<=month[d]<=e]
    if cand: sel[lab]=max(cand,key=ndvi_mean)
fig,ax=plt.subplots(figsize=(12,5))
for lab,d in sel.items():
    a,nd,_,_=read_date(d); m=~nd
    means=[np.nanmean(a[m,b]) for b in range(len(S2_BANDS))]
    ax.plot(range(len(S2_BANDS)),means,'o-',lw=2,ms=8,label=f'{lab} ({d})')
ax.set_xticks(range(len(S2_BANDS))); ax.set_xticklabels(S2_BANDS,fontsize=8)
ax.set_ylabel('Reflektansi rata-rata'); ax.legend(fontsize=9); ax.grid(alpha=.3)
ax.set_title('Profil Spektral — Tanggal NDVI Tertinggi per Kuartal (2024)'); show(fig,'s2_spectral_profile.png')

## 10. Band correlation matrix (`s2_band_correlation.png`)

Streams ~50k pixels sampled across all dates.

In [ ]:
N_TOTAL=50000; B=len(S2_BANDS); per=max(1,N_TOTAL//len(DATES)); rng=np.random.default_rng(0)
chunks=[]
for d in DATES:
    a,_,_,_=read_date(d); flat=a.reshape(-1,B)
    ok=np.all(flat>0,1)&np.all(np.isfinite(flat),1); flat=flat[ok]
    if len(flat)==0: continue
    k=min(per,len(flat)); chunks.append(flat[rng.choice(len(flat),k,replace=False)])
sample=np.vstack(chunks); corr=np.corrcoef(sample.T)
fig,ax=plt.subplots(figsize=(9,8)); im=ax.imshow(corr,cmap='RdBu_r',vmin=-1,vmax=1)
ax.set_xticks(range(B)); ax.set_yticks(range(B)); ax.set_xticklabels(S2_BANDS); ax.set_yticklabels(S2_BANDS)
plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
for i in range(B):
    for j in range(B):
        ax.text(j,i,f'{corr[i,j]:.2f}',ha='center',va='center',fontsize=7.5,
                color='white' if abs(corr[i,j])>0.7 else 'black')
plt.colorbar(im,ax=ax,shrink=.8).set_label('Korelasi Pearson')
ax.set_title(f'Korelasi Antar-Band (~{len(sample):,} piksel)'); show(fig,'s2_band_correlation.png')

## 11. Per-class band histograms, peak date (`s2_band_histograms_per_class_row*.png`)

Justifies percentile [2,98] normalization — heavy-tailed reflectance.

In [ ]:
arr,nd,tf,crs=read_date(PEAK_DATE); cdl=load_cdl_grid(nd.shape,tf,crs); valid=~nd
SAMPLE=200_000; rng=np.random.default_rng(42)
def samp(v): return v if len(v)<=SAMPLE else v[rng.choice(len(v),SAMPLE,replace=False)]
for row in range(0,len(S2_BANDS),5):
    bands=S2_BANDS[row:row+5]
    fig,axes=plt.subplots(1,len(bands),figsize=(4.2*len(bands),4))
    axes=np.atleast_1d(axes)
    for ax,band in zip(axes,bands):
        i=S2_BANDS.index(band); bd=arr[:,:,i]; allv=samp(bd[valid]); lo,hi=np.percentile(allv,[.5,99.5])
        ax.hist(allv,bins=150,range=(lo,hi),color='#b3aee0',alpha=.5,label='Seluruh piksel',density=True)
        for cid,cn in TRAIN_CLASSES.items():
            mk=valid&(cdl==cid)
            if mk.sum()<50: continue
            vv=samp(bd[mk]); vv=vv[(vv>=lo)&(vv<=hi)]
            if len(vv)<50: continue
            ax.hist(vv,bins=150,range=(lo,hi),histtype='step',color=CDL_COLORS.get(cid,'#333'),lw=1.3,label=cn,density=True)
        ax.set_title(f'{band} ({BAND_LABELS[band]})',fontsize=9); ax.set_xlabel('Reflektansi'); ax.set_ylabel('Densitas')
        ax.axvline(np.percentile(allv,2),color='red',ls='--',lw=1,alpha=.7); ax.axvline(np.percentile(allv,98),color='red',ls='--',lw=1,alpha=.7)
        if row==0 and band==bands[0]: ax.legend(fontsize=6,loc='upper right',ncol=2)
    plt.tight_layout(); show(fig,f's2_band_histograms_per_class_row{row//5+1}.png')

## 12. S2 + CDL patch detail (`s2_cdl_patch_detail.png`)

Two 256×256 patches: true-color S2 vs CDL labels.

In [ ]:
PS=256; s2_path=DATE2FILE[PEAK_DATE]
with rasterio.open(s2_path) as src: H,W,tf,crs=src.height,src.width,src.transform,src.crs
cdlg=load_cdl_grid((H,W),tf,crs)
CRGB={k:mcolors.to_rgb(v) for k,v in CDL_COLORS.items()}; KEEP=set(CRGB)
def scan(req,seed):
    rng=np.random.default_rng(seed); out=[]
    for _ in range(4000):
        r=rng.integers(2,H//PS-2)*PS; c=rng.integers(2,W//PS-2)*PS
        cp=cdlg[r:r+PS,c:c+PS]; cls=set(np.unique(cp))&KEEP-{0}
        if len(cls)<3 or np.mean(cp==0)>.3: continue
        if req is not None and np.mean(cp==req)<.2: continue
        with rasterio.open(s2_path) as src: b0=src.read(1,window=Window(c,r,PS,PS)).astype(np.float32)
        if np.mean((b0>-9990)&(b0!=0)&~np.isnan(b0))>.9: out.append((r,c,len(cls)))
        if len(out)>=15: break
    return sorted(out,key=lambda x:-x[2])
rice=scan(3,42); ra,ca,_=rice[0]
gen=scan(None,43); rb,cb=next(((r,c) for r,c,_ in gen if abs(r-ra)>PS*4 or abs(c-ca)>PS*4),(gen[1][0],gen[1][1]))
def mk_rgb(a):
    nd=(a[:,:,0]<=-9990)|(a[:,:,0]==0)|np.isnan(a[:,:,0])
    def n(x):
        vv=x[~nd]; vv=vv[~np.isnan(vv)]; lo,hi=np.percentile(vv,[2,98])
        o=np.clip((x-lo)/(hi-lo+1e-6),0,1); o[nd]=0; return o
    return np.stack([n(a[:,:,S2_BANDS.index('B4')]),n(a[:,:,S2_BANDS.index('B3')]),n(a[:,:,S2_BANDS.index('B2')])],-1)
def cdl_rgb(cp):
    im=np.ones((*cp.shape,3),np.float32)*.83
    for cls,col in CRGB.items(): im[cp==cls]=col
    return im
fig,axes=plt.subplots(2,2,figsize=(10,10))
for i,(r,c) in enumerate([(ra,ca),(rb,cb)]):
    with rasterio.open(s2_path) as src: a=src.read(window=Window(c,r,PS,PS)).astype(np.float32).transpose(1,2,0)
    cp=cdlg[r:r+PS,c:c+PS]
    axes[i,0].imshow(mk_rgb(a)); axes[i,0].set_title(f'Patch {chr(65+i)} — True Color',fontsize=14); axes[i,0].axis('off')
    axes[i,1].imshow(cdl_rgb(cp)); axes[i,1].set_title(f'Patch {chr(65+i)} — CDL 2024',fontsize=14); axes[i,1].axis('off')
    leg=[mpatches.Patch(color=CRGB.get(x,(.83,.83,.83)),label=TRAIN_CLASSES.get(x,'bg' if x==0 else f'cls {x}')) for x in np.unique(cp) if x in CDL_COLORS]
    axes[i,1].legend(handles=leg,loc='lower right',fontsize=8,framealpha=.85)
plt.tight_layout(); show(fig,'s2_cdl_patch_detail.png')